# 09 — Subplots & Layouts
**Goal:** Compose multi-panel figures — uniform grids, irregular layouts via
`GridSpec`, small multiples, and the new `subplot_mosaic` API.

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

np.random.seed(44)
print('numpy', np.__version__)

## 1. The three layout patterns

1. **Uniform grid** — `plt.subplots(2, 3)` — every cell same size.
2. **GridSpec** — variable cell sizes, controlled widths/heights.
3. **subplot_mosaic** — ASCII-art layout; the most readable for complex
   dashboards.

## 2. Uniform grid — `plt.subplots(nrows, ncols)`

The simplest and most common case. Always pass `figsize` and consider
`sharex` / `sharey` when the panels share a scale.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 6), sharex=True, sharey=True)
for ax in axes.ravel():
    ax.plot(np.cumsum(np.random.randn(50)))
    ax.spines[['top', 'right']].set_visible(False)
fig.suptitle('Uniform 2×3 grid', weight='bold')
plt.tight_layout(); plt.show()

## 3. `subplot_mosaic` — readable complex layouts

Mosaic is the cleanest way to express an irregular layout. Each character is a
panel; repeated characters mean the same panel spans cells.

In [ ]:
fig, axd = plt.subplot_mosaic(
    [
        ['A', 'A', 'B'],
        ['C', 'D', 'D'],
        ['C', 'E', 'E'],
    ],
    figsize=(11, 6),
)
for k, ax in axd.items():
    ax.plot(np.cumsum(np.random.randn(60)))
    ax.set_title(k, weight='bold')
    ax.spines[['top', 'right']].set_visible(False)
fig.suptitle('Mosaic — A spans 2 cols, D/E span 2 rows, C spans 2 rows', weight='bold')
plt.tight_layout(); plt.show()

## 4. `GridSpec` — for very fine control

Use GridSpec when you need explicit `width_ratios` or `height_ratios`,
or to leave a cell empty (`add_subplot` with a slice that overlaps a
non-existent neighbor is awkward).

In [ ]:
fig = plt.figure(figsize=(11, 5))
gs = GridSpec(2, 3, figure=fig, width_ratios=[1, 1, 1.2],
              height_ratios=[1, 1.5], wspace=0.35, hspace=0.4)
ax1 = fig.add_subplot(gs[0, 0]); ax1.plot([1, 2, 3], [1, 4, 2])
ax2 = fig.add_subplot(gs[0, 1]); ax2.bar(['a','b','c'], [3, 1, 5])
ax3 = fig.add_subplot(gs[0, 2]); ax3.scatter(np.random.rand(20), np.random.rand(20))
ax4 = fig.add_subplot(gs[1, :2])  # spans 2 columns
ax4.plot(np.cumsum(np.random.randn(50)), color='crimson')
ax5 = fig.add_subplot(gs[1, 2])
ax5.hist(np.random.randn(500), bins=30, color='steelblue', edgecolor='white')
for ax in [ax1, ax2, ax3, ax4, ax5]:
    ax.spines[['top', 'right']].set_visible(False)
fig.suptitle('GridSpec — explicit width/height ratios', weight='bold')
plt.show()

## 5. Small multiples — the same chart, repeated

The single most useful pattern in this notebook. The principle: same scale,
same geometry, one panel per group.

In [ ]:
channels = ['Search', 'Social', 'Display', 'Email']
fig, axes = plt.subplots(2, 2, figsize=(10, 7), sharex=True, sharey=True)
for ax, ch in zip(axes.ravel(), channels):
    series = np.cumsum(np.random.randn(60)) + np.random.uniform(0, 10)
    ax.plot(series, color='steelblue')
    ax.fill_between(range(len(series)), series, alpha=0.2, color='steelblue')
    ax.set_title(ch, weight='bold')
    ax.spines[['top', 'right']].set_visible(False)
fig.suptitle('Small multiples — same scale, same geometry', weight='bold')
plt.tight_layout(); plt.show()

## 6. Shared scales — when and how

- `sharex=True` / `sharey=True` — same numeric range, easier comparison.
- `sharex='col'`, `sharex='row'` — share only within a column/row.
- Be explicit when scales differ — log, percent, or absolute values rarely
  belong on the same axis.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for i, s in enumerate([True, False]):
    for j in range(4):
        axes[i].plot(np.cumsum(np.random.randn(50)))
    axes[i].set_title(f'sharey={s}')
    axes[i].spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.show()

## 7. Insets — a zoom inside a chart

Use `axes.inset_axes` to drop a small chart inside a panel. The companion
`indicate_inset_zoom` highlights the zoomed region on the main chart.

In [ ]:
x = np.linspace(0, 10, 500)
y = np.sin(x) + 0.1 * np.random.randn(500)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(x, y, color='steelblue')
ax.spines[['top', 'right']].set_visible(False)

axins = ax.inset_axes([0.55, 0.55, 0.4, 0.4])
axins.plot(x, y, color='steelblue')
axins.set_xlim(4, 6); axins.set_ylim(-1.3, 1.3)
axins.set_xticklabels([]); axins.set_yticklabels([])
ax.indicate_inset_zoom(axins, edgecolor='crimson')
ax.set_title('Inset — zoom into x ∈ [4, 6]')
plt.tight_layout(); plt.show()

## 8. Twin axes — for related but differently-scaled series

Two y-axes on the same x. **Use sparingly** — it is often misread. The
common legitimate case: a *rate* overlaid on a *count* (e.g. conversion rate
on top of clicks).

In [ ]:
days = pd.date_range('2024-01-01', periods=90, freq='D')
clicks = np.cumsum(np.random.randint(80, 200, 90)) + 5000
cvr    = np.clip(np.random.normal(0.04, 0.005, 90), 0, 0.1)

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(days, clicks, color='steelblue', alpha=0.4, label='Clicks')
ax.set_ylabel('Clicks', color='steelblue')
ax.tick_params(axis='y', labelcolor='steelblue')
ax2 = ax.twinx()
ax2.plot(days, cvr, color='crimson', lw=2, label='CVR')
ax2.set_ylabel('Conversion rate', color='crimson')
ax2.tick_params(axis='y', labelcolor='crimson')
ax2.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
ax.spines[['top']].set_visible(False)
ax2.spines[['top']].set_visible(False)
ax.set_title('Clicks (count) + CVR (rate) — twin axis')
plt.tight_layout(); plt.show()

## 9. Real data — KPI dashboard panel

Build a small 4-panel KPI summary using the unified marketing data.

In [ ]:
df = pd.read_csv('data/clean/unified_daily.csv')
num = df.select_dtypes('number').columns.tolist()
print(num[:6])
kpi = num[:4] if len(num) >= 4 else num
fig, axes = plt.subplots(2, 2, figsize=(11, 6.5))
for ax, col in zip(axes.ravel(), kpi):
    s = df[col].dropna()
    ax.hist(s, bins=25, color='steelblue', edgecolor='white')
    ax.set_title(col)
    ax.spines[['top', 'right']].set_visible(False)
fig.suptitle('KPI distribution panel', weight='bold')
plt.tight_layout(); plt.show()

## Summary

| Pattern | Use |
|---|---|
| `plt.subplots(n, m)` | Uniform grid, same scale |
| `subplot_mosaic` | Readable irregular layouts |
| `GridSpec` | Explicit width/height ratios |
| Small multiples | Same chart, one per group, shared scale |
| `inset_axes` | Zoom-in callouts |
| `twinx` | Two differently-scaled series (use sparingly) |

**Next:** `10_annotations_storytelling.ipynb` — making the chart argue a point.